# Phase0 Semantic Explorer (JSON-Only)

This notebook explores the input JSON as a semantic graph.


## What This Notebook Does

- Reads one input JSON file from `data/`.
- Normalizes the zone data locally in the notebook.
- Builds an RDF graph with zones, walls, windows, and properties.
- Runs SPARQL queries for quick inspection.
- Exports graph files and visualizes only the first `NORTH` zone.

It does **not** import `phase0` modules and does **not** connect to IDA ICE.


In [10]:
from pathlib import Path
import json

import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF
from rdflib.namespace import XSD

try:
    from pyvis.network import Network
    HAS_PYVIS = True
except Exception:
    HAS_PYVIS = False


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for p in [current] + list(current.parents):
        if (p / 'data').exists() and (p / 'Semantic Enrichment').exists():
            return p
    raise RuntimeError('Could not locate repository root from current path.')


REPO_ROOT = find_repo_root(Path.cwd())
print(f'Repo root: {REPO_ROOT}')


Repo root: C:\Users\luis.sanchez\Documents\00_SANCHEZ-EQUA\00_MY_RESOURCES\00_REPOSITORIES\PHAERO_2_ESBO_v2


In [11]:
# Input JSON path (change this if needed)
INPUT_JSON = REPO_ROOT / 'data' / 'example_case_v2.sample.json'
print('Input JSON:', INPUT_JSON)


def normalize_zones_from_json(path: Path):
    raw = json.loads(path.read_text(encoding='utf-8'))

    # v2 compact format: shared + zones (NORTH/SOUTH/EAST/WEST/INTERNAL_ONLY)
    if isinstance(raw, dict) and isinstance(raw.get('shared'), dict) and isinstance(raw.get('zones'), dict):
        shared = raw['shared']
        geometry = shared.get('geometry', {})
        window_defaults = shared.get('window_defaults', {})
        wall_cons_default = shared.get('wall_constructions', {})

        zone_type = str(shared.get('zone_type', '')).strip()
        zones = []

        wall_by_orientation = {
            'NORTH': 'WALL_1',
            'SOUTH': 'WALL_2',
            'EAST': 'WALL_3',
            'WEST': 'WALL_4',
        }

        for orientation, cfg in raw['zones'].items():
            cfg = cfg or {}
            zname = f"{zone_type}_{orientation}" if zone_type else orientation

            wwr_external = float(cfg.get('wwr_external', 0.0) or 0.0)
            ext_wall = wall_by_orientation.get(orientation)

            wwr = {f'WALL_{i}': 0.0 for i in range(1, 5)}
            if ext_wall and orientation != 'INTERNAL_ONLY':
                wwr[ext_wall] = wwr_external

            # Start with neutral defaults; allow explicit per-zone overrides.
            surface_part = {
                f'WALL_{i}': {'internal_fraction': 0.5, 'side': 'left'}
                for i in range(1, 5)
            }
            for wall_name, sp in (cfg.get('surface_part') or {}).items():
                if wall_name in surface_part and isinstance(sp, dict):
                    surface_part[wall_name].update(sp)

            zone = {
                'zone_name': zname,
                'zone_type': zone_type,
                'zone_multiplier': float(cfg.get('zone_multiplier', 1.0) or 1.0),
                'room_length': float(geometry.get('room_length', 0.0) or 0.0),
                'room_width': float(geometry.get('room_width', 0.0) or 0.0),
                'room_height': float(geometry.get('room_height', 0.0) or 0.0),
                'wwr': wwr,
                'wall_constructions': cfg.get('wall_constructions', wall_cons_default),
                'surface_part': surface_part,
                'glazing_type': cfg.get('glazing_type', window_defaults.get('glazing_type', '')),
                'frame_area': float(cfg.get('frame_area', window_defaults.get('frame_area', 0.0)) or 0.0),
                'frame_u_value': float(cfg.get('frame_u_value', window_defaults.get('frame_u_value', 0.0)) or 0.0),
                'shading_type': cfg.get('shading_type', window_defaults.get('shading_type', '')),
            }
            zones.append(zone)

        return zones

    # Legacy fallback: already-expanded list under common keys.
    if isinstance(raw, dict):
        for key in ['zone_configs', 'zones_expanded', 'zones_list']:
            if isinstance(raw.get(key), list):
                return raw[key]

    if isinstance(raw, list):
        return raw

    raise ValueError('Unsupported JSON structure for semantic explorer notebook.')


a_zones = normalize_zones_from_json(INPUT_JSON)
print(f'Expanded zones: {len(a_zones)}')
print('Sample zone names:', [z.get('zone_name', '<missing>') for z in a_zones[:3]])


Input JSON: C:\Users\luis.sanchez\Documents\00_SANCHEZ-EQUA\00_MY_RESOURCES\00_REPOSITORIES\PHAERO_2_ESBO_v2\data\example_case_v2.sample.json
Expanded zones: 5
Sample zone names: ['1_NORTH', '1_SOUTH', '1_EAST']


In [12]:
# Build RDF graph from normalized zone configs

EX = Namespace('http://example.org/phaero#')
BEM = Namespace('https://example.com/bem#')
BRICK = Namespace('https://brickschema.org/schema/Brick#')

g = Graph()
g.bind('ex', EX)
g.bind('bem', BEM)
g.bind('brick', BRICK)

def literal_num(v):
    return Literal(float(v), datatype=XSD.double)

for zone in a_zones:
    zname = zone['zone_name']
    zuri = EX[zname]

    g.add((zuri, RDF.type, BRICK.Room))
    g.add((zuri, BEM.zoneName, Literal(zname)))
    g.add((zuri, BEM.zoneType, Literal(str(zone.get('zone_type', '')))))
    g.add((zuri, BEM.zoneMultiplier, literal_num(zone.get('zone_multiplier', 1))))

    # Orientation from suffix
    orientation = zname.split('_')[-1]
    g.add((zuri, BEM.orientation, Literal(orientation)))

    # Geometry
    g.add((zuri, BEM.roomLength, literal_num(zone.get('room_length', 0))))
    g.add((zuri, BEM.roomWidth, literal_num(zone.get('room_width', 0))))
    g.add((zuri, BEM.roomHeight, literal_num(zone.get('room_height', 0))))

    # Window system per zone
    ws_uri = EX[f'{zname}_WINDOW_SYSTEM']
    g.add((ws_uri, RDF.type, BEM.WindowSystem))
    g.add((ws_uri, BEM.glazingType, Literal(str(zone.get('glazing_type', '')))))
    g.add((ws_uri, BEM.frameArea, literal_num(zone.get('frame_area', 0))))
    g.add((ws_uri, BEM.frameUValue, literal_num(zone.get('frame_u_value', 0))))
    g.add((ws_uri, BEM.shadingType, Literal(str(zone.get('shading_type', '')))))

    walls = zone.get('wwr', {})
    wall_cons = zone.get('wall_constructions', {})
    surface = zone.get('surface_part', {})

    for wall_name in ['WALL_1', 'WALL_2', 'WALL_3', 'WALL_4']:
        wuri = EX[f'{zname}_{wall_name}']
        g.add((wuri, RDF.type, BRICK.Wall))
        g.add((zuri, BEM.hasSurface, wuri))
        g.add((wuri, BEM.wwr, literal_num(walls.get(wall_name, 0))))

        wc = wall_cons.get(wall_name, {})
        g.add((wuri, BEM.internalConstruction, Literal(str(wc.get('internal', '')))))
        g.add((wuri, BEM.externalConstruction, Literal(str(wc.get('external', '')))))

        sp = surface.get(wall_name, {})
        part_uri = EX[f'{zname}_{wall_name}_SUBPART']
        g.add((wuri, BEM.hasSubpart, part_uri))
        g.add((part_uri, BEM.internalFraction, literal_num(sp.get('internal_fraction', 0))))
        g.add((part_uri, BEM.side, Literal(str(sp.get('side', 'left')))))

        if float(walls.get(wall_name, 0)) > 0:
            win_uri = EX[f'{zname}_{wall_name}_WINDOW']
            g.add((win_uri, RDF.type, BEM.Window))
            g.add((wuri, BEM.hasOpening, win_uri))
            g.add((win_uri, BEM.WWR, literal_num(walls.get(wall_name, 0))))
            g.add((win_uri, BEM.usesWindowSystem, ws_uri))

print(f'Graph triples: {len(g)}')


Graph triples: 241


In [13]:
# Optional export and visualization
out_ttl = REPO_ROOT / 'Semantic Enrichment' / 'phase0_semantic_graph.ttl'
out_jsonld = REPO_ROOT / 'Semantic Enrichment' / 'phase0_semantic_graph.jsonld'
g.serialize(out_ttl, format='turtle')
g.serialize(out_jsonld, format='json-ld', indent=2)
print('Saved:', out_ttl)
print('Saved:', out_jsonld)

# Visualize only the first NORTH zone subgraph.
first_north = None
for s, _, _ in g.triples((None, BEM.orientation, Literal('NORTH'))):
    first_north = s
    break

if first_north is None:
    print('No NORTH zone found; skipped visualization.')
elif HAS_PYVIS:
    zone_token = str(first_north).split('#')[-1]
    net = Network(height='750px', width='100%', directed=True)

    for s, p, o in g:
        s_str, o_str = str(s), str(o)
        if zone_token in s_str or zone_token in o_str:
            p_str = str(p)
            net.add_node(s_str, label=s_str.split('#')[-1])
            net.add_node(o_str, label=o_str.split('#')[-1])
            net.add_edge(s_str, o_str, label=p_str.split('#')[-1])

    html_path = REPO_ROOT / 'Semantic Enrichment' / 'phase0_semantic_graph_north.html'
    net.write_html(str(html_path), open_browser=False)
    print('Saved NORTH-only visualization:', html_path)
else:
    print('pyvis not installed; skipped HTML network output.')


Saved: C:\Users\luis.sanchez\Documents\00_SANCHEZ-EQUA\00_MY_RESOURCES\00_REPOSITORIES\PHAERO_2_ESBO_v2\Semantic Enrichment\phase0_semantic_graph.ttl
Saved: C:\Users\luis.sanchez\Documents\00_SANCHEZ-EQUA\00_MY_RESOURCES\00_REPOSITORIES\PHAERO_2_ESBO_v2\Semantic Enrichment\phase0_semantic_graph.jsonld
Saved NORTH-only visualization: C:\Users\luis.sanchez\Documents\00_SANCHEZ-EQUA\00_MY_RESOURCES\00_REPOSITORIES\PHAERO_2_ESBO_v2\Semantic Enrichment\phase0_semantic_graph_north.html


## Query 1: Zone multipliers by orientation


In [14]:
q1 = '''
PREFIX bem: <https://example.com/bem#>
PREFIX brick: <https://brickschema.org/schema/Brick#>

SELECT ?zone ?orientation ?multiplier
WHERE {
  ?zone a brick:Room .
  ?zone bem:orientation ?orientation .
  ?zone bem:zoneMultiplier ?multiplier .
}
ORDER BY ?zone
'''

rows = []
for r in g.query(q1):
    rows.append({
        'zone': str(r.zone).split('#')[-1],
        'orientation': str(r.orientation),
        'multiplier': float(r.multiplier),
    })

pd.DataFrame(rows)


,zone,orientation,multiplier
0,1_EAST,EAST,2.0
1,1_INTERNAL_ONLY,ONLY,2.0
2,1_NORTH,NORTH,2.0
3,1_SOUTH,SOUTH,2.0
4,1_WEST,WEST,2.0


## Query 2: Walls and WWR


In [15]:
q2 = '''
PREFIX bem: <https://example.com/bem#>
PREFIX brick: <https://brickschema.org/schema/Brick#>

SELECT ?wall ?wwr
WHERE {
  ?wall a brick:Wall .
  ?wall bem:wwr ?wwr .
}
ORDER BY ?wall
'''

rows = []
for r in g.query(q2):
    rows.append({'wall': str(r.wall).split('#')[-1], 'wwr': float(r.wwr)})

pd.DataFrame(rows)


,wall,wwr
0,1_EAST_WALL_1,0.0
1,1_EAST_WALL_2,0.0
2,1_EAST_WALL_3,0.3
3,1_EAST_WALL_4,0.0
4,1_INTERNAL_ONLY_WALL_1,0.0
5,1_INTERNAL_ONLY_WALL_2,0.0
6,1_INTERNAL_ONLY_WALL_3,0.0
7,1_INTERNAL_ONLY_WALL_4,0.0
8,1_NORTH_WALL_1,0.5
9,1_NORTH_WALL_2,0.0


## Query 3: Construction types per wall


In [16]:
q3 = '''
PREFIX bem: <https://example.com/bem#>
PREFIX brick: <https://brickschema.org/schema/Brick#>

SELECT ?wall ?internal ?external
WHERE {
  ?wall a brick:Wall .
  ?wall bem:internalConstruction ?internal .
  ?wall bem:externalConstruction ?external .
}
ORDER BY ?wall
'''

rows = []
for r in g.query(q3):
    rows.append({
        'wall': str(r.wall).split('#')[-1],
        'internal': str(r.internal),
        'external': str(r.external),
    })

pd.DataFrame(rows)


,wall,internal,external
0,1_EAST_WALL_1,IW_TB,AW_BE_MW
1,1_EAST_WALL_2,IW_TB,AW_BE_MW
2,1_EAST_WALL_3,IW_TB,AW_BE_MW
3,1_EAST_WALL_4,IW_TB,AW_BE_MW
4,1_INTERNAL_ONLY_WALL_1,IW_TB,AW_BE_MW
5,1_INTERNAL_ONLY_WALL_2,IW_TB,AW_BE_MW
6,1_INTERNAL_ONLY_WALL_3,IW_TB,AW_BE_MW
7,1_INTERNAL_ONLY_WALL_4,IW_TB,AW_BE_MW
8,1_NORTH_WALL_1,IW_TB,AW_BE_MW
9,1_NORTH_WALL_2,IW_TB,AW_BE_MW


## Query 4: Window systems (glazing, frame, shading)


In [17]:
q4 = '''
PREFIX bem: <https://example.com/bem#>

SELECT ?system ?glazing ?frame_area ?frame_u ?shading
WHERE {
  ?system a bem:WindowSystem .
  ?system bem:glazingType ?glazing .
  ?system bem:frameArea ?frame_area .
  ?system bem:frameUValue ?frame_u .
  ?system bem:shadingType ?shading .
}
ORDER BY ?system
'''

rows = []
for r in g.query(q4):
    rows.append({
        'system': str(r.system).split('#')[-1],
        'glazing': str(r.glazing),
        'frame_area': float(r.frame_area),
        'frame_u': float(r.frame_u),
        'shading': str(r.shading),
    })

pd.DataFrame(rows)


,system,glazing,frame_area,frame_u,shading
0,1_EAST_WINDOW_SYSTEM,Double Clear Air 2-panes,23.0,1.0,OUTSIDE-BLIND
1,1_INTERNAL_ONLY_WINDOW_SYSTEM,Double Clear Air 2-panes,23.0,1.0,OUTSIDE-BLIND
2,1_NORTH_WINDOW_SYSTEM,Double Clear Air 2-panes,23.0,1.0,OUTSIDE-BLIND
3,1_SOUTH_WINDOW_SYSTEM,Double Clear Air 2-panes,23.0,1.0,OUTSIDE-BLIND
4,1_WEST_WINDOW_SYSTEM,Double Clear Air 2-panes,23.0,1.0,OUTSIDE-BLIND


## Query 5: Internal fraction map (wall subparts)


In [18]:
q5 = '''
PREFIX bem: <https://example.com/bem#>
PREFIX brick: <https://brickschema.org/schema/Brick#>

SELECT ?wall ?fraction ?side
WHERE {
  ?wall a brick:Wall .
  ?wall bem:hasSubpart ?part .
  ?part bem:internalFraction ?fraction .
  ?part bem:side ?side .
}
ORDER BY ?wall
'''

rows = []
for r in g.query(q5):
    rows.append({
        'wall': str(r.wall).split('#')[-1],
        'internal_fraction': float(r.fraction),
        'side': str(r.side),
    })

pd.DataFrame(rows)


,wall,internal_fraction,side
0,1_EAST_WALL_1,0.5,left
1,1_EAST_WALL_2,0.5,left
2,1_EAST_WALL_3,0.5,left
3,1_EAST_WALL_4,0.5,left
4,1_INTERNAL_ONLY_WALL_1,1.0,left
5,1_INTERNAL_ONLY_WALL_2,1.0,left
6,1_INTERNAL_ONLY_WALL_3,1.0,left
7,1_INTERNAL_ONLY_WALL_4,1.0,left
8,1_NORTH_WALL_1,0.5,left
9,1_NORTH_WALL_2,0.5,left
